# Curtain (1-panel) — Water Age animation
**Adapted from** `csiem-marvl/custom_py/tfv_curtain_panel/curtain_1panel.ipynb`.

Reads the **1.7.0 SH 2023B** WQ run and animates `WQ_TRC_AGE` (water age) along the
Cockburn Sound N–S curtain line, with depth-averaged velocity vectors overlaid.

Only the model file pointer, output filename, and run label differ from the original.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import seaborn as sns
import geopandas as gpd
import pandas as pd
import tfv.xarray
import cmocean  # 'cmo.*' colormaps (berlin needs mpl>=3.10; not available here)
import xarray as xr
from tfv.extractor import FvExtractor
from tfv.timeseries import FvTimeSeries
from shapely import wkt
# A nice plot default
sns.set(style='white', font_scale=0.8)


In [ ]:
# --- Model file (1.7.0 SH 2023B WQ run) ---
model_folder = Path(r'W:\WAMSI\1.7\SH-20251123-1.7.0\2023B-20251124150126\results')
model_file = 'csiem_B010_20221101_20240401_WQ_WQ.nc'
ds = xr.open_dataset(model_folder / model_file, decode_times=True, engine='netcdf4')
fv = ds.tfv
fv


In [ ]:
# --- Curtain polyline (N-S section through Cockburn Sound) ---
# Relative to this notebook in custom_py/tfv_curtain_panel/
shp_path = r"../../gis/Curtain/New_Curtain_line_LL_100m.shp"
gdf = gpd.read_file(shp_path)
# Make sure the CRS is geographic (lat/lon)
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

gdf["longitude"] = gdf.geometry.x
gdf["latitude"]  = gdf.geometry.y

df = gdf.drop(columns="geometry")
polyline = df[['longitude', 'latitude']].to_numpy()
print(f"Curtain polyline: {polyline.shape[0]} vertices")


In [ ]:
from matplotlib import animation

# --- Output setup ---
output_folder = Path('./outputs')
output_folder.mkdir(exist_ok=True)
out_file_gif = output_folder / 'curtain_1panel_2023B_AGE.gif'

# --- Time range for animation (within run window 2022-11-01 .. 2024-04-01) ---
time_start = '2023-01-25 00:00'
time_end   = '2023-03-25 23:00'
fvx = fv.sel(Time=slice(time_start, time_end))

RUN_LABEL = 'CSIEM-1.7.0 2023B'

# --- Plot style ---
sns.set_theme(style='white', font_scale=0.8)

# --- Figure: single curtain axis ---
fig = plt.figure(figsize=(12, 3.6), constrained_layout=True)
ax_sal_curtain = fig.add_subplot(1, 1, 1)
ax_sal_curtain.set_facecolor('grey')

# --- Initial frame (frame 0) ---
sal_curtain = fvx.plot_curtain(
    polyline, "WQ_TRC_AGE", 0, ax=ax_sal_curtain,
    ec="face", cmap="cmo.matter", clim=(5e5, 3e6), colorbar=False  # sequential age ramp (orig used 'berlin')
)
cbar_sal = fig.colorbar(
    sal_curtain.patch, ax=ax_sal_curtain,
    orientation="vertical", fraction=0.025, pad=0.02
)
cbar_sal.set_ticks([604800, 1209600, 1814400, 2419200, 2937600])
cbar_sal.set_ticklabels(['7', '14', '21', '28', '34'])
cbar_sal.set_label("Age (days)")

sal_vec = fvx.plot_curtain_vector(
    polyline, 0, ax=ax_sal_curtain,
    tangential=False, scale=5, color="w", width=0.001
)

# --- Format curtain axis ---
ax_sal_curtain.set_title('Water "Age"')
xticks = [1000, 6000, 17060, 25000, 35000]
xticklabels = ['Sepia Depression', 'Causeway', 'Central Basin', 'Parmelia Bank', 'Fremantle']
ax_sal_curtain.set_xticks(xticks)
ax_sal_curtain.set_xticklabels(xticklabels, rotation=0, fontsize=9)
ax_sal_curtain.set_xlabel("Transect Location")
ax_sal_curtain.text(0.02, 1.05, 'S', transform=ax_sal_curtain.transAxes,
                    fontsize=12, fontweight='bold', va='bottom')
ax_sal_curtain.text(0.98, 1.05, 'N', transform=ax_sal_curtain.transAxes,
                    fontsize=12, fontweight='bold', va='bottom', ha='right')

# --- Animation function ---
def animate(i):
    sal_curtain.set_time_current(i)
    sal_vec.set_time_current(i)
    date = sal_curtain.get_time_current()
    fig.suptitle(f'{RUN_LABEL} - {date.strftime("%Y-%m-%d %H:%M")}', fontweight='bold')

nframes = fvx.sizes['Time']
print(f"Creating animation with {nframes} frames...")
anim = animation.FuncAnimation(fig, animate, frames=nframes, repeat=False, interval=200)

print(f"Saving animation to {out_file_gif}...")
writer = animation.PillowWriter(fps=3)
anim.save(out_file_gif, writer=writer, dpi=100)
print(f"Animation saved successfully to {out_file_gif}")
